## Task-adaptive pretraining (TAPT)

### Colab Setup

In [2]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

Mounted at /content/drive


### Key Imports

In [3]:
import torch

from config import APT, APT_EPOCHS, RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.apt_pools import build_tapt_pool
from data.loader_twd_labelled import load_splits
from models.apt import adapt
from models.plm_finetune import finetune
from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
SEEDS = SHAH_SEEDS
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


### Continued pretraining (one adapted model per seed)

In [4]:
# TAPT pretrains on the task's own training sentences, labels ignored.
# budget and optimiser settings come from config.APT (Gururangan Table 13).
# one model per seed, since
# each seed has a different train split -- unlike DAPT, which is seed-independent.
ARM = "tapt:roberta-large"
ENC = "roberta-large"
EPOCHS = APT_EPOCHS["tapt"]

for seed in SEEDS:
    save_dir = str(RESULTS_DIR / "models" / f"tapt-s{seed}")
    if os.path.isdir(save_dir):
        print(f"{ARM} seed {seed}: already adapted, skipping")
        continue
    train, _ = load_splits("benchmark", seed=seed)
    sentences = train["sentence"].to_list()
    print(
        f"{ARM} seed {seed}: {len(sentences):,} sentences, {EPOCHS} epochs", flush=True
    )
    adapt(
        sentences,
        model_name=SHAH_PLM[ENC]["model_name"],
        epochs=EPOCHS,
        save_dir=save_dir,
        device=DEVICE,
        verbose=True,
        **APT,
    )

Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
tapt:roberta-large seed 5768: 1,984 sentences, 100 epochs


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

    saved -> /content/drive/MyDrive/thesis/models/tapt-s78516
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
tapt:roberta-large seed 944601: 1,984 sentences, 100 epochs


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

    tokenizing 1,984 sentences...
    training: 62 steps/epoch x 100 epoch(s)
    epoch 0: mlm loss 1.3300
    epoch 1: mlm loss 1.2010
    epoch 2: mlm loss 1.2302
    epoch 3: mlm loss 1.1902
    epoch 4: mlm loss 1.1250
    epoch 5: mlm loss 1.1068
    epoch 6: mlm loss 1.0612
    epoch 7: mlm loss 0.9908
    epoch 8: mlm loss 1.0181
    epoch 9: mlm loss 1.0272
    epoch 10: mlm loss 0.9580
    epoch 11: mlm loss 0.9563
    epoch 12: mlm loss 0.9317
    epoch 13: mlm loss 0.8977
    epoch 14: mlm loss 0.9007
    epoch 15: mlm loss 0.8938
    epoch 16: mlm loss 0.9581
    epoch 17: mlm loss 0.9359
    epoch 18: mlm loss 0.9068
    epoch 19: mlm loss 0.8517
    epoch 20: mlm loss 0.8371
    epoch 21: mlm loss 0.8107
    epoch 22: mlm loss 0.7985
    epoch 23: mlm loss 0.8248
    epoch 24: mlm loss 0.7767
    epoch 25: mlm loss 0.7799
    epoch 26: mlm loss 0.7742
    epoch 27: mlm loss 0.7329
    epoch 28: mlm loss 0.7117
    epoch 29: mlm loss 0.6958
    epoch 30: mlm loss 0.7243
  

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    saved -> /content/drive/MyDrive/thesis/models/tapt-s944601


### Curated-TAPT: continued pretraining on the filtered corpus

In [ ]:
# curated-TAPT pretrains on the pool the labelled set was sampled from, 35,257
# filtered sentences minus that seed's test split. Gururangan section 5.1.
CURATED_ARM = "curated-tapt:roberta-large"
CURATED_EPOCHS = APT_EPOCHS["curated-tapt"]

for seed in SEEDS:
    save_dir = str(RESULTS_DIR / "models" / f"curated-tapt-s{seed}")
    if os.path.isdir(save_dir):
        print(f"{CURATED_ARM} seed {seed}: already adapted, skipping")
        continue
    sentences = build_tapt_pool(seed)["sentence"].tolist()
    print(
        f"{CURATED_ARM} seed {seed}: {len(sentences):,} sentences, "
        f"{CURATED_EPOCHS} epochs",
        flush=True,
    )
    adapt(
        sentences,
        model_name=SHAH_PLM[ENC]["model_name"],
        epochs=CURATED_EPOCHS,
        save_dir=save_dir,
        device=DEVICE,
        verbose=True,
        **APT,
    )


### Fine-tune adapted encoders (3 seeds)

In [ ]:
cfg = SHAH_PLM[ENC]

for arm, dirname in [(ARM, "tapt"), (CURATED_ARM, "curated-tapt")]:
    for seed in SEEDS:
        if already_done(OUT, force=FORCE, model=arm, corpus="twd", seed=seed):
            print(f"{arm} seed {seed}: already done, skipping")
            continue
        train, test = load_splits("benchmark", seed=seed)
        model, tok_, metrics = finetune(
            train,
            model_name=str(RESULTS_DIR / "models" / f"{dirname}-s{seed}"),
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=arm,
            corpus="twd",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{arm} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()
